Directory three of the project:

```
├── code
├── data
├── jmxs
├── model
├── pcaps
└── results
```

The working dir of the script is the code/ directory

The code will be used to extract data to instrument the model. The data has been obtained from the system executing the application. The perform N readings in a observation time interval T. 

    1. read data from the data dir/ - 
    
    ```
    ├── 10-38
    │   ├── containers.json
    │   ├── energy.csv
    │   ├── perf.json
    │   └── requests.jtl
    ├── 11-38
    │   ├── containers.json
    │   ├── energy.csv
    │   ├── perf.json
    │   └── requests.jtl
    ```
       
    The name of the directory contains two numbers. The former describes the ith reading while the second one the population of customers operating while reading performance and energy values.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import csv, json, glob, os

In [2]:
CUSTOMERS = 38
DATA = f"../model/results"
DIRS = os.listdir(DATA)

In [3]:
paths = {
    'energy'     : [f'{DATA}/{x}/energy.csv' for x in DIRS],
    'containers' : [f'{DATA}/{x}/containers.json' for x in DIRS],
    'performance': [f'{DATA}/{x}/perf.json' for x in DIRS]
}

In [4]:
def get_energy_stats(trials):
    time   = np.array([len(x['time']) for x in trials])
    energy = np.array(
        [np.trapz(x['power'], x['time']) for x in trials]
    )
    e = energy/time
    
    print(
            f"# Energy Per Visit(Joule/Visit):\n",
            f"## Mean:\t\t\t{energy.mean()}", 
            f"## Min-Max:\t\t\t[{energy.min()}, {energy.max()}]",
            f"## Var:\t\t\t\t{energy.var()}", 
            f"## Std:\t\t\t\t{energy.std()}", 
            '\n'
            f"# Average Response Time:\t{time.mean()}",
            f"# e (Joule/s):\t\t\t{e.mean()}, [{e.min()}, {e.max()}]",
            sep='\n'
         )

In [5]:
energy_values = [
    pd.read_csv(x, names = ["time", "power"])
    for x in paths.get('energy')
]

In [6]:
get_energy_stats(energy_values)

# Energy Per Visit(Joule/Visit):

## Mean:			290.79561333333334
## Min-Max:			[247.67814999999996, 312.2742]
## Var:				430.44829411065587
## Std:				20.747247868347642

# Average Response Time:	5.8
# e (Joule/s):			50.130743555555554, [49.43851666666668, 52.045700000000004]


In [7]:
from stats import SystemStats

In [8]:
perf_values = [
    pd.DataFrame(SystemStats(x).get_stats()) for x in paths.get('performance')
]

In [9]:
from stats import ContainerStats